In [1]:
import pandas as pd
import numpy as np
import ast

In [2]:
# Load files
credits = pd.read_csv('/Users/jakelee/Desktop/Movie Recommendation/tmdb_5000_credits.csv')
movies = pd.read_csv('/Users/jakelee/Desktop/Movie Recommendation/tmdb_5000_movies.csv')

In [3]:
# View file 'credits'
credits.head(1)

,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."


In [4]:
# View file 'credits'
# Extract useful columns
movies = movies[['id','genres','keywords','overview','tagline','vote_average']]
movies.head(1)

,id,genres,keywords,overview,tagline,vote_average
0,19995,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","In the 22nd century, a paraplegic Marine is di...",Enter the World of Pandora.,7.2


In [5]:
# Merge 2 data sets
movies = pd.merge(credits, movies, left_on="movie_id", right_on="id")

In [6]:
# drop duplicate column
movies = movies.drop('id',axis=1)
movies.head(1)

,movie_id,title,cast,crew,genres,keywords,overview,tagline,vote_average
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","In the 22nd century, a paraplegic Marine is di...",Enter the World of Pandora.,7.2


In [7]:
# use ats to parse string to list for readability
for c in ['cast', 'crew', 'genres', 'keywords']:
    movies[c] = movies[c].apply(ast.literal_eval)

In [8]:
# Extract only top 3 actors
def get_top3_actors(cast):
    top3_actors = []
    for actor in cast:
        if actor['order'] in [0,1,2]:
            top3_actors.append(actor['name'])
    return top3_actors

In [9]:
movies['cast'] = movies['cast'].apply(get_top3_actors)

In [10]:
movies.head(2)

,movie_id,title,cast,crew,genres,keywords,overview,tagline,vote_average
0,19995,Avatar,"[Sam Worthington, Zoe Saldana, Sigourney Weaver]","[{'credit_id': '52fe48009251416c750aca23', 'de...","[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...","[{'id': 1463, 'name': 'culture clash'}, {'id':...","In the 22nd century, a paraplegic Marine is di...",Enter the World of Pandora.,7.2
1,285,Pirates of the Caribbean: At World's End,"[Johnny Depp, Orlando Bloom, Keira Knightley]","[{'credit_id': '52fe4232c3a36847f800b579', 'de...","[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...","[{'id': 270, 'name': 'ocean'}, {'id': 726, 'na...","Captain Barbossa, long believed to be dead, ha...","At the end of the world, the adventure begins.",6.9


In [11]:
# Extract only director
movies['crew'][0]

def get_director(crew):
    director = []
    for role in crew:
        if role['job'] == 'Director':
            director.append(role['name'])
            break
    return director

movies['crew'] = movies['crew'].apply(get_director)

In [12]:
# Rename 'crew' -> 'director'
movies = movies.rename(columns={'crew': 'director'})
movies.head(2)

,movie_id,title,cast,director,genres,keywords,overview,tagline,vote_average
0,19995,Avatar,"[Sam Worthington, Zoe Saldana, Sigourney Weaver]",[James Cameron],"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...","[{'id': 1463, 'name': 'culture clash'}, {'id':...","In the 22nd century, a paraplegic Marine is di...",Enter the World of Pandora.,7.2
1,285,Pirates of the Caribbean: At World's End,"[Johnny Depp, Orlando Bloom, Keira Knightley]",[Gore Verbinski],"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...","[{'id': 270, 'name': 'ocean'}, {'id': 726, 'na...","Captain Barbossa, long believed to be dead, ha...","At the end of the world, the adventure begins.",6.9


In [13]:
movies['genres'][0]

[{'id': 28, 'name': 'Action'},
 {'id': 12, 'name': 'Adventure'},
 {'id': 14, 'name': 'Fantasy'},
 {'id': 878, 'name': 'Science Fiction'}]

In [14]:
# Extract 'name' from 'genres' and 'keywords'
def get_names(col):
    names = []
    for c in col:
        names.append(c['name'].lower())
    return names

movies['genres'] = movies['genres'].apply(get_names)
movies['keywords'] = movies['keywords'].apply(get_names)

In [15]:
movies['genres'][0]

['action', 'adventure', 'fantasy', 'science fiction']

In [16]:
movies['keywords'][0]

['culture clash',
 'future',
 'space war',
 'space colony',
 'society',
 'space travel',
 'futuristic',
 'romance',
 'space',
 'alien',
 'tribe',
 'alien planet',
 'cgi',
 'marine',
 'soldier',
 'battle',
 'love affair',
 'anti war',
 'power relations',
 'mind and soul',
 '3d']

In [17]:
# Create tags
movies['tags'] = movies['genres'] + movies['keywords'] + movies['cast'] + movies['director']

In [18]:
# Replace 'space' with ''
movies['tags'] = movies['tags'].apply(lambda x: [i.replace(" ", "") for i in x])
movies['tags'][0]

['action',
 'adventure',
 'fantasy',
 'sciencefiction',
 'cultureclash',
 'future',
 'spacewar',
 'spacecolony',
 'society',
 'spacetravel',
 'futuristic',
 'romance',
 'space',
 'alien',
 'tribe',
 'alienplanet',
 'cgi',
 'marine',
 'soldier',
 'battle',
 'loveaffair',
 'antiwar',
 'powerrelations',
 'mindandsoul',
 '3d',
 'SamWorthington',
 'ZoeSaldana',
 'SigourneyWeaver',
 'JamesCameron']

In [19]:
# covert list to single string for model
movies['tags'] = movies['tags'].apply(lambda x: " ".join(x))
movies['tags'][0]

'action adventure fantasy sciencefiction cultureclash future spacewar spacecolony society spacetravel futuristic romance space alien tribe alienplanet cgi marine soldier battle loveaffair antiwar powerrelations mindandsoul 3d SamWorthington ZoeSaldana SigourneyWeaver JamesCameron'

In [21]:
# turn text into vectors
from sklearn.feature_extraction.text import CountVectorizer
vectorizer = CountVectorizer(max_features=5000, stop_words='english')
vectors = vectorizer.fit_transform(movies['tags']).toarray()
print(vectors)  


[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]


In [22]:
# Find similarity between movies
from sklearn.metrics.pairwise import cosine_similarity
similarity = cosine_similarity(vectors)
print(similarity[0])

'''Meaning:

* 1.0 → identical
* close to 1 → very similar
* close to 0 → different'''

[1.         0.12309149 0.11111111 ... 0.06085806 0.         0.        ]


'Meaning:\n\n* 1.0 → identical\n* close to 1 → very similar\n* close to 0 → different'

In [23]:
# Rank and return 10 similar movies
def recommend(movie):
    index = movies[movies['title'].str.lower().str.strip() == movie].index[0]
    scores = similarity[index]
    movie_list = sorted(list(enumerate(scores)), reverse=True, key= lambda x: x[1])[1:11]
    titles = []
    for i in movie_list:
        title = movies.iloc[i[0]].title
        titles.append(title)
    return titles 

In [24]:
# save files
# pickle.dump(your_data, open('filename.pkl', 'wb'))

import pickle

pickle.dump(movies, open('movies.pkl', 'wb'))
pickle.dump(similarity, open('model.pkl', 'wb'))